# IFT 6758 - Devoir 2

In [ ]:
%load_ext autoreload
%autoreload 2

## Question 1

### a)
Commencez par utiliser les fonctions créées dans `q1.py` pour rendre les données plus informatives et lisibles. Concrètement, remplissez les cellules suivantes :


In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from q1 import count_labels, convert_id, convert_ids, contains_label, get_correlation

sns.set(style="ticks")

In [ ]:
# Charger le fichier `audio_segments.csv` dans un DataFrame `df`
df_Audio_csv = pd.read_csv('./data/audio_segments.csv')



In [ ]:
# Ajouter une colonne correspondant au nombre d'étiquettes appelée `label_count`
# Ici, il y a un problème ennuyeux avec l'accès à la colonne positive_labels

df_Audio_csv.columns = df_Audio_csv.columns.str.lstrip("# ") # sinon il ya un espace au debut du nom de chaquue colonne


df_Audio_csv['label_count'] = df_Audio_csv['positive_labels'].apply(count_labels) # On creer une nvelle colonne ou chaque elem sera l image de l element de la case de positive labels via la fct


In [ ]:
# Ajouter une nouvelle colonne appelée `label_names` avec les noms d'étiquette traités au lieu de l'ID d'étiquette

# Imprimer le temps pris pour cette opération (soit en utilisant le module time ou timeit).
# Puisque nous n'exécutons ce code qu'une fois, ce n'est pas très problématique que cela prenne quelques minutes.
# Cependant, pour un ensemble de données plus volumineux, cela vaudrait la peine de l'accélérer
# (par exemple en créant un dictionnaire ID -> nom une fois et en l'utilisant).

t0 = time.time()
df_Audio_csv['label_names'] = df_Audio_csv['positive_labels'].apply(convert_ids)
t1 = time.time()

print(f"Temps pris: {t1 - t0:.2f} secondes")



In [ ]:
# Affichez le DataFrame et enregistrez-le dans `audio_segments_clean.csv` (sans index)
df_Audio_csv.to_csv("data/audio_segments_clean.csv", index=False)
df_Audio_csv.head()




### b)

Ensuite, à l'aide du DataFrame propre, remplissez les cellules suivantes pour mieux comprendre la distribution des étiquettes dans l'ensemble de données. Pour chaque graphique ci-dessous, assurez-vous d'inclure les **noms d'axe** appropriés et un **titre**.

In [ ]:
# À l'aide de seaborn, créez un histogramme du nombre d'étiquettes des rangées dans le DataFrame
sns.histplot(data=df_Audio_csv, x="label_count")


Suivez les étapes ci-dessous pour créer un heatmap montrant la "corrélation" entre différentes étiquettes.
- Plus précisément, chaque cellule de la heatmap doit correspondre à la probabilité qu'un échantillon avec l'étiquette de ligne correspondante ait également l'étiquette de colonne correspondante.
- Considérez simplement les étiquettes ["Piano", "Classical music", "Speech", "Conversation", "Screaming"].

Votre graphique final devrait ressembler à ceci :

![alt text](images/heatmap.png "Heatmap")

In [ ]:
labels = ["Piano", "Music", "Speech", "Conversation", "Screaming"]

# Il y a plusieurs façons d'aborder cela, la façon que nous recommandons ici est de construire d'abord une grille 2D où chaque
# value est la valeur de corrélation entre la ligne/colonne correspondante à l'aide des fonctions créées dans q1.py.

matrice2Dcorr = np.zeros((len(labels), len(labels))) #matrice 5x5
for i in range(len(labels)):
    for j  in range(len(labels)):
        matrice2Dcorr[i,j] = get_correlation(df_Audio_csv['label_names'],labels[i],labels[j])

        
# Ensuite, à l'aide de sns.heatmap, créez la heatmap, en profitant de xticklabels et yticklabels pour définir les noms des étiquettes comme valeurs de graduation
mat_conf = sns.heatmap(matrice2Dcorr,yticklabels =labels,  xticklabels=labels, annot = True,cmap='YlOrRd')


## Question 2
La question 2 n'a pas de composante notebook, remplissez simplement le fichier `q2.py`.

## Question 3

Téléchargez l'audio pour les étiquettes suivantes à l'aide des fonction créées dans `q3.py`

In [ ]:
from q3 import data_pipeline, rename_files, filter_df

In [ ]:
# Téléchargez "Cough"

data_pipeline("data/audio_segments_clean.csv", "Cough")

# Renommez les fichiers pour inclure le début et fin des échantillons ainsi que la durée
rename_files("audio/Cough_cut","data/audio_segments_clean.csv")



In [ ]:
# Téléchargez "Hammer"

data_pipeline("data/audio_segments_clean.csv", "Hammer")


# Renommez les fichiers pour inclure le début et fin des échantillons ainsi que la durée

rename_files("audio/Hammer_cut","data/audio_segments_clean.csv")

Comme vous l'avez probablement remarqué, le téléchargement de toutes ces données audio est lent (et en tant que tel, nous vous avons uniquement demandé de télécharger 2 des étiquettes). Dans de nombreux cas, il est possible d'obtenir des augmentations de performances significatives en utilisant soit le multiprocessing (https://docs.python.org/3/library/multiprocessing.html) soit le multithreading (https://docs.python.org/3/library/threading.html)  qui pourrait par exemple vous permettre de télécharger plusieurs fichiers audio en parallèle.

En règle générale, utilisez le multithreading lorsque vos programmes sont bloqués par l'IO (par exemple ici) et le multiprocessing lorsqu'ils sont bloqués CPU (et utilisez ainsi tous les cores de votre CPU).

## Question 4
Pour les cellules suivantes, utilisez l'ID "0GNNFBrRz1E". Complétez les fonctions et exécutez les 
 fournies ci-dessous.

In [ ]:
from IPython.display import Audio
import librosa


In [ ]:
# Jouez le segment audio dans le notebook en utilisant
# https://ipython.org/ipython-doc/dev/api/generated/IPython.display.html#IPython.display.Audio


path = "audio/Hammer_cut/0GNNFBrRz1E_40_50_10.mp3"
y, sr = librosa.load(path, sr=None)  # sr=None pour garder le sample rate d'origine

# 2. Jouer le son dans le notebook
Audio(y, rate=sr)

Une façon de visualiser l'audio consiste à utiliser des spectrogrammes mel. Brièvement, les spectrogrammes Mel convertissent l'audio en une image 2D grâce à l'utilisation de Fourier Transforms (plus de détails peuvent être trouvés ici: https://medium.com/analytics-vidhya/understanding-the-mel-spectrogram-fca2afa2ce53).

In [ ]:
stft_hopsize = 128
n_fft = 512
sample_rate = 16000

def to_log_scale(mel: np.ndarray) -> np.ndarray:
    mel = np.log(mel + 1e-6)/2.0
    return mel

def create_mel_spectrogram(mp3_path: str) -> np.ndarray:
    """ 

    En utilisant librosa (https://librosa.org/doc/main/generated/librosa.feature.melspectrogram.html) écrivez une fonction qui:
    1. Charge l'audio à partir d'un mp3_path (en utilisant librosa)
    2. Le convertit en un spectrogramme mel (en utilisant les paramètres fournis ci-dessus)
    3. Applique la transformation d'échelle logarithmique au spectrogramme mel (fourni ci-dessus une fois de plus)
    4. Renvoie le spectrogramme mel transformé

    Assurez-vous de passer le sample rate
    """
    y, sr = librosa.load(mp3_path, sr=sample_rate, mono=True)

    # 2) mel-spectrogram (puissance)
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=n_fft,
        hop_length=stft_hopsize,
        power=2.0,        # puissance (magnitude^2), valeur par défaut
        n_mels=128,       # standard courant; ajuste si ton autograder précise autre chose
        center=True
    )

    # 3) log-scale
    mel_log = to_log_scale(mel)

    # 4) retour (float32 utile pour downstream)
    return mel_log.astype(np.float32)
    

M = create_mel_spectrogram("audio/Hammer_cut/0GNNFBrRz1E_40_50_10.mp3")


Les données audio peuvent également être visualisées en regardant la forme d'onde (c'est-à-dire sous la forme d'un tracé linéaire des valeurs d'amplitude). Nous combinerons les deux méthodes de visualisation ci-dessous. Le graphique résultant devrait ressembler à :
![alt text](images/combined_plot.png "Combined Plot")

In [ ]:
def plot_audio(mp3_path: str) -> None:
    """ 
    En utilisant matplotlib et create_mel_spectrogram() écrivez une fonction qui prend un mp3_path et trace
    à la fois la forme d'onde (graphique linéaire des amplitudes) et le spectrogramme mel côte à côte en tant que subplots.

    Utilisez le mp3_path comme titre principal unique pour tout le graphique
    """

    plt.figure()
    plt.imshow(M, origin="lower", aspect="auto")
    plt.title("Mel spectrogram (log)")
    plt.xlabel("Frames")
    plt.ylabel("Mel bins")
    plt.show()
        

plot_audio("audio/Hammer_cut/0GNNFBrRz1E_40_50_10.mp3")